In [1]:
import pandas as pd
import numpy as np
import warnings, re
import matplotlib 
warnings.filterwarnings('ignore')

In [2]:
bouts = pd.read_csv('../data/sumodb_all_bouts_final.csv')
players = pd.read_csv('../data\sumodb_names_batch_20250923_105158\sumodb_wrestlers_final.csv')

In [3]:
bouts.head(10)

,basho,day,wrestler1,wrestler1_rank,wrestler1_result,wrestler2,wrestler2_rank,wrestler2_result,kimarite,scraped_at,source_offset
0,2000.01,1,Aminishiki,J13w,1-0 (8-7),Wakakosho,J13e,0-1 (8-7),yorikiri,2025-09-22T18:52:24.380838,0
1,2000.01,1,Kitazakura,J12w,1-0 (7-8),Takamisakari,J12e,0-1 (7-8),yorikiri,2025-09-22T18:52:24.380838,0
2,2000.01,1,Tochinohana,J11e,1-0 (9-6),Kobo,J11w,0-1 (7-8),yoritaoshi,2025-09-22T18:52:24.380838,0
3,2000.01,1,Tamanonada,J10w,1-0 (5-10),Ganyu,J10e,0-1 (7-8),yorikiri,2025-09-22T18:52:24.380838,0
4,2000.01,1,Tomonohana,J9e,1-0 (9-6),Sentoryu,J9w,0-1 (7-8),shitatedashinage,2025-09-22T18:52:24.380838,0
5,2000.01,1,Gojoro,J8e,1-0 (7-7-1),Jumonji,J8w,0-1 (8-7),oshidashi,2025-09-22T18:52:24.380838,0
6,2000.01,1,Dewataira,J7w,1-0 (7-8),Kotomitsuki,J7e,0-1 (11-4),uwatenage,2025-09-22T18:52:24.380838,0
7,2000.01,1,Akinoshu,J6w,1-0 (10-5),Susanoumi,J6e,0-1 (8-7),katasukashi,2025-09-22T18:52:24.380838,0
8,2000.01,1,Kitakachidoki,J5w,1-0 (9-6),Daishi,J5e,0-1 (5-10),oshidashi,2025-09-22T18:52:24.380838,0
9,2000.01,1,Dewaarashi,J4w,1-0 (7-8),Mitoizumi,J4e,0-1 (8-7),yorikiri,2025-09-22T18:52:24.380838,0


In [4]:
players.head(10)

,rikishi,heya,shusshin,birth_date,hatsu,intai,height,weight,highest_rank,career_high,date_info,rank_info,age_info,record_info,scraped_at,source_offset
0,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1966.11,Sd91w,18.1,2-5,2025-09-23T10:52:07.501629,100000
1,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.01,Jd15e,19.0,4-3,2025-09-23T10:52:07.502630,100000
2,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.03,Sd91e,19.2,3-4,2025-09-23T10:52:07.502630,100000
3,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.05,Jd59e,19.4,5-2,2025-09-23T10:52:07.502630,100000
4,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.07,Jd5w,19.6,1-6,2025-09-23T10:52:07.502630,100000
5,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.09,Jd46e,19.8,5-2,2025-09-23T10:52:07.502630,100000
6,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.11,Jd6w,19.1,6-1,2025-09-23T10:52:07.502630,100000
7,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,178.0,78.0,Sd66,Ms41,1968.01,Sd66e,20.0,1-6,2025-09-23T10:52:07.502630,100000
8,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,178.0,78.0,Sd66,Ms41,1968.03,Sd98e,20.2,4-3,2025-09-23T10:52:07.502630,100000
9,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,178.0,78.0,Sd66,Ms41,1968.05,Sd74e,20.4,3-4,2025-09-23T10:52:07.502630,100000


In [5]:
# Filter the ongoing September tournament
bouts = bouts[bouts['basho']!=2025.09]

# Create a stacked bouts that shows win vs loss. Right now, wrestler1 is always the winner
bouts['bout_id'] = np.arange(len(bouts))

bouts_win = bouts[['basho', 'day', 'wrestler1', 'wrestler1_rank', 'wrestler1_result',
       'wrestler2', 'wrestler2_rank', 'wrestler2_result', 'kimarite', 'bout_id']]
bouts_win['win'] = 1
bouts_loss = bouts[['basho', 'day', 'wrestler1', 'wrestler1_rank', 'wrestler1_result',
       'wrestler2', 'wrestler2_rank', 'wrestler2_result', 'kimarite', 'bout_id']]
bouts_loss['win'] = 0

bouts_win.columns = ['basho', 'day', 'name', 'rank', 'tournament_record', 'opponent_name', 'opponent_rank', 'opponent_tournament_record', 'technique', 'bout_id', 'win']
bouts_loss.columns = ['basho', 'day', 'opponent_name', 'opponent_rank', 'opponent_tournament_record', 'name', 'rank', 'tournament_record', 'technique', 'bout_id', 'win']

bouts_all = pd.concat([bouts_win, bouts_loss], ignore_index=True)

In [6]:
# Removing low-ranked players
players = players[~players['rank_info'].isin(['Mz','Bg'])] # Mae-zumo and Banzuke-gai

players = players[~players['rank_info'].str.startswith('Jd', na=False)] # Jonidan
players = players[~players['rank_info'].str.startswith('Jk', na=False)] # Jonokuchi
players = players[~players['rank_info'].str.startswith('Sd', na=False)] # Sandanme

players['division'] = np.where(players['rank_info'].str.startswith('Ms', na=False), 'Makushita',
                      np.where(players['rank_info'].str.startswith('J', na=False), 'Juryo',
                      np.where(players['rank_info'].str.startswith('M', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('Y', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('K', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('S', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('O', na=False), 'Maegashira', ''
                      )))))))

In [7]:
# Create experience field
players['hatsu'] = players['hatsu'].astype(str).str.replace('.','/')
players['date2'] = players['date_info'].astype(str).str.replace('.','/')

players['debut_date'] = pd.to_datetime(players['hatsu'], errors='coerce', format='%Y/%m')
players['date2']       = pd.to_datetime(players['date2'] , errors='coerce', format='%Y/%m')
players['experience'] = (players['date2'] - players['debut_date']).dt.days / 365
players.drop(columns=['debut_date','date2'], inplace=True)

players['birth_date'] = pd.to_datetime(players['birth_date'], errors='coerce', format='%d.%m.%Y')

players.rename(columns={
    'date_info'   : 'date', 
    'rank_info'   : 'rank',
    'age_info'    : 'age', 
    'record_info' : 'record'
}, inplace=True)

In [8]:
# Creating a cross-division rank
def extract_letters_numbers(s):
    letters = re.search(r'[A-Za-z]+', str(s))
    numbers = re.search(r'\d+', str(s))
    return (letters.group(0) if letters else None, numbers.group(0) if numbers else None)

players[['rank_letters', 'rank_numbers']] = players['rank'].apply(lambda x: pd.Series(extract_letters_numbers(x)))

# Point of this modifier is to adjust the within-division ranks to reflect the number of ranks in the higher-ranking divisions
players['rank_letter_modifier'] = np.where(
    players['rank_letters'] == 'Y', 0, np.where(
    players['rank_letters'] == 'O', 2, np.where(
    players['rank_letters'] == 'S', 5, np.where(
    players['rank_letters'] == 'K', 7, np.where(
    players['rank_letters'] == 'M', 9, np.where(
    players['rank_letters'] == 'J', 27, np.where(
    players['rank_letters'] == 'Ms', 41, 999)))))))

# convert to numeric
players['rank_numbers'] = pd.to_numeric(players['rank_numbers'], errors='coerce')
players['rank_num'] = players['rank_numbers'] + players['rank_letter_modifier']
players.drop(columns=['rank_letters', 'rank_numbers', 'rank_letter_modifier'], inplace=True)

In [9]:
# Merge the player information on bouts_all
players1 = players.copy()
players2 = players.copy()
players2.columns = [col if col in ['rikishi', 'date','rank'] else 'opponent_'+col for col in players2.columns]

df_all = pd.merge(bouts_all, players1, left_on=['basho','name','rank'], right_on=['date','rikishi','rank'], how='left', validate='m:1', suffixes=(None,'_y'))
df_all.drop(columns=['date','rikishi'], inplace=True)
df_all = pd.merge(df_all, players2, left_on=['basho','opponent_name','opponent_rank'], right_on=['date','rikishi','rank'], how='left', validate='m:1', suffixes=(None,'_y'))
df_all.drop(columns=['date','rikishi','rank_y'], inplace=True)

In [10]:
df_all = df_all[[
    'basho', 'day', 'bout_id', 'technique', 'win',
    'name', 'heya', 'shusshin', 'birth_date', 'age', 'division', 
    'height', 'weight', 'experience', 'rank', 'rank_num', 
    'opponent_name', 'opponent_heya', 'opponent_shusshin', 'opponent_age', 'opponent_division', 
    'opponent_height', 'opponent_weight', 'opponent_experience', 'opponent_rank','opponent_rank_num'
]]

In [11]:
# Create columns comparing player to opponent
df_all['age_diff'] = df_all['age'] - df_all['opponent_age']
df_all['height_diff'] = df_all['height'] - df_all['opponent_height']
df_all['weight_diff'] = df_all['weight'] - df_all['opponent_weight']
df_all['experience_diff'] = df_all['experience'] - df_all['opponent_experience']
df_all['rank_diff'] = df_all['rank_num'] - df_all['opponent_rank_num']
df_all['flag_higher_division'] = np.where(
    (df_all['division'] == 'Maegashira') & (df_all['opponent_division'] == 'Juryo')    , 1, np.where(
    (df_all['division'] == 'Juryo')      & (df_all['opponent_division'] == 'Makushita'), 1, 0
    ))
df_all['flag_lower_division'] = np.where(
    (df_all['division'] == 'Juryo')     & (df_all['opponent_division'] == 'Maegashira'), 1, np.where(
    (df_all['division'] == 'Makushita') & (df_all['opponent_division'] == 'Juryo')     , 1, 0
    ))
df_all['flag_same_hometown'] = np.where(df_all['shusshin'] == df_all['opponent_shusshin'], 1, 0)
df_all['flag_first_day'] = np.where(df_all['day'] == 1, 1, 0)
df_all['flag_last_day'] = np.where(df_all['day'] >= 15, 1, 0)

In [12]:
# Create historical performance statistics
# because wrestlers can have the same names, we identify using multiple variables

# Calculate cumulative win rate for each wrestler up to each bout
df_all.sort_values(by=['name','shusshin','birth_date','basho','day','bout_id'], inplace=True)
df_all['cum_wins'] = df_all.groupby(['name','shusshin','birth_date'])['win'].cumsum() - df_all['win']
df_all['cum_bouts'] = df_all.groupby(['name','shusshin','birth_date']).cumcount()
df_all['cum_win_rate'] = np.where(df_all['cum_bouts'] > 0, df_all['cum_wins'] / df_all['cum_bouts'], np.nan)

# Calculate win rate for previous 6 tournaments for each wrestler
df_all['basho_dt'] = pd.to_datetime(df_all['basho'].astype(str), format='%Y.%m')
df_all.sort_values(by=['name','shusshin','birth_date','basho_dt','day','bout_id'], inplace=True)

# Compute win rate per tournament
tournament_stats = df_all.groupby(['name','shusshin','birth_date','basho','basho_dt']).agg(
    wins=('win','sum'),
    bouts=('win','count')
).reset_index()

# Rolling aggregate for previous 6 tournaments (excluding current)
tournament_stats['wins_prev6'] = tournament_stats.groupby(['name','shusshin','birth_date'])['wins'].transform(
    lambda x: x.shift(1).rolling(window=6, min_periods=1).sum()
)
tournament_stats['bouts_prev6'] = tournament_stats.groupby(['name','shusshin','birth_date'])['bouts'].transform(
    lambda x: x.shift(1).rolling(window=6, min_periods=1).sum()
)
tournament_stats['win_rate_prev6'] = np.where(
    tournament_stats['bouts_prev6'] > 0,
    tournament_stats['wins_prev6'] / tournament_stats['bouts_prev6'],
    np.nan
)

# Merge back to df_all
df_all = pd.merge(df_all, tournament_stats[['name','shusshin','birth_date', 'basho', 'wins_prev6', 'bouts_prev6', 'win_rate_prev6']],
                  on=['name','shusshin','birth_date','basho'], how='left')

df_all.drop(columns=['basho_dt'], inplace=True)

In [13]:
# Now we need to join the win rate fields for opponents as well
winrate_opp = df_all[['name', 'bout_id', 'cum_wins', 'cum_bouts',
       'cum_win_rate', 'wins_prev6', 'bouts_prev6',
       'win_rate_prev6']]

winrate_opp.columns = [col if col in ['bout_id'] else 'opponent_'+col for col in winrate_opp.columns]

df_all = pd.merge(df_all, winrate_opp, on=['opponent_name','bout_id'], how='left', validate='1:1')

In [14]:
df_all = df_all.dropna()
df_all.to_pickle('../data/cleaned_sumo_bouts.pkl')
df_all.head(10)

,basho,day,bout_id,technique,win,name,heya,shusshin,birth_date,age,...,cum_win_rate,wins_prev6,bouts_prev6,win_rate_prev6,opponent_cum_wins,opponent_cum_bouts,opponent_cum_win_rate,opponent_wins_prev6,opponent_bouts_prev6,opponent_win_rate_prev6
16,2015.05,2,45227,hatakikomi,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,216,438,0.493151,40.0,77.0,0.519481
17,2015.05,3,45264,sukuinage,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.529412,7.0,15.0,0.466667,83,165,0.503030,40.0,83.0,0.481928
18,2015.05,4,45295,tsukiotoshi,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,3,6,0.500000,2.0,3.0,0.666667
19,2015.05,5,45329,uwatenage,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.473684,7.0,15.0,0.466667,11,26,0.423077,8.0,22.0,0.363636
20,2015.05,6,45365,hatakikomi,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,117,266,0.439850,33.0,76.0,0.434211
21,2015.05,7,45397,oshidashi,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.523810,7.0,15.0,0.466667,0,1,0.000000,0.0,1.0,0.000000
22,2015.05,8,45433,hikiotoshi,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,20,37,0.540541,15.0,30.0,0.500000
23,2015.05,9,45466,oshidashi,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.521739,7.0,15.0,0.466667,14,24,0.583333,9.0,16.0,0.562500
24,2015.05,10,45501,hatakikomi,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,117,224,0.522321,44.0,92.0,0.478261
25,2015.05,11,45534,oshitaoshi,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.480000,7.0,15.0,0.466667,204,431,0.473318,38.0,90.0,0.422222


In [15]:
# Create an aggregated df by tournament/basho
df_all = df_all[[
    'basho','name','heya','shusshin','age', 'division', 
    'height', 'weight', 'experience', 'rank_num',
    'win', 'opponent_rank_num', 'age_diff', 'height_diff', 'weight_diff',
    'experience_diff', 'rank_diff', 'flag_higher_division',
    'flag_lower_division', 'flag_same_hometown', 'cum_wins', 'cum_bouts',
    'cum_win_rate', 'wins_prev6', 'bouts_prev6', 'win_rate_prev6',
    'opponent_cum_wins', 'opponent_cum_bouts', 'opponent_cum_win_rate',
    'opponent_wins_prev6', 'opponent_bouts_prev6',
    'opponent_win_rate_prev6'
]]

df_tournament = df_all.groupby(['basho','name','heya','shusshin','age', 'division', 
    'height', 'weight', 'experience', 'rank_num']).agg(['mean', 'sum', 'count']).reset_index()

# Flatten the names
df_tournament.columns = ['_'.join(col).strip() for col in df_tournament.columns.values]

df_tournament = df_tournament[[
    'basho_', 'name_', 'heya_', 'shusshin_', 'age_', 'division_', 'height_',
    'weight_', 'experience_', 'rank_num_', 'win_mean', 'win_sum',
    'win_count', 'opponent_rank_num_mean', 'age_diff_mean', 'height_diff_mean', 'weight_diff_mean', 'experience_diff_mean', 'rank_diff_mean', 'flag_higher_division_mean',
    'flag_lower_division_mean', 'flag_same_hometown_mean',
    'cum_wins_mean','cum_bouts_mean', 'cum_win_rate_mean','wins_prev6_mean','bouts_prev6_mean','win_rate_prev6_mean', 'opponent_cum_wins_mean', 'opponent_cum_bouts_mean', 'opponent_cum_win_rate_mean',
    'opponent_wins_prev6_mean', 'opponent_bouts_prev6_mean',
    'opponent_win_rate_prev6_mean'
]]

df_tournament.columns = [
    'basho', 'name', 'heya', 'shusshin', 'age', 'division', 'height',
    'weight', 'experience', 'rank_num', 'win_mean', 'win_sum',
    'win_count', 'opponent_rank_num_mean', 'age_diff_mean', 'height_diff_mean', 'weight_diff_mean', 'experience_diff_mean', 'rank_diff_mean', 'flag_higher_division_mean',
    'flag_lower_division_mean', 'flag_same_hometown_mean',
    'cum_wins_mean','cum_bouts_mean', 'cum_win_rate_mean','wins_prev6_mean','bouts_prev6_mean','win_rate_prev6_mean', 'opponent_cum_wins_mean', 'opponent_cum_bouts_mean', 'opponent_cum_win_rate_mean',
    'opponent_wins_prev6_mean', 'opponent_bouts_prev6_mean',
    'opponent_win_rate_prev6_mean'
]

In [16]:
# Filter to remove injuries/early leaving of tournament (< 7 bouts)
df_tournament = df_tournament[df_tournament['win_count']>=7]

In [17]:
df_tournament = df_tournament.dropna()
df_tournament.to_pickle('../data/cleaned_sumo_tournaments.pkl')
df_tournament.head(10)

,basho,name,heya,shusshin,age,division,height,weight,experience,rank_num,...,cum_win_rate_mean,wins_prev6_mean,bouts_prev6_mean,win_rate_prev6_mean,opponent_cum_wins_mean,opponent_cum_bouts_mean,opponent_cum_win_rate_mean,opponent_wins_prev6_mean,opponent_bouts_prev6_mean,opponent_win_rate_prev6_mean
0,2000.03,Akebono,Azumazeki,U.S.A.,30.10,Maegashira,203.5,230.5,12.008219,1.0,...,0.742232,11.0,15.0,0.733333,13.000000,22.000000,0.588693,8.866667,15.000000,0.591111
1,2000.03,Akinoshima,Futagoyama,Hiroshima,32.11,Maegashira,175.0,156.5,18.013699,14.0,...,0.449472,7.0,15.0,0.466667,10.466667,21.266667,0.497952,7.266667,14.266667,0.508889
2,2000.03,Akinoshu,Izutsu,Hiroshima,31.10,Juryo,179.0,118.5,15.506849,29.0,...,0.579146,10.0,15.0,0.666667,10.466667,22.000000,0.485061,7.466667,15.000000,0.497778
3,2000.03,Aminishiki,Ajigawa,Aomori,21.50,Juryo,184.0,113.5,3.164384,35.0,...,0.535114,8.0,15.0,0.533333,10.200000,22.000000,0.470059,6.600000,15.000000,0.440000
4,2000.03,Asanosho,Wakamatsu,Kanagawa,30.20,Maegashira,185.5,144.5,8.005479,19.0,...,0.461973,7.0,15.0,0.466667,11.000000,22.000000,0.505768,7.600000,15.000000,0.506667
5,2000.03,Asanowaka,Wakamatsu,Aichi,30.30,Maegashira,176.0,142.3,8.005479,17.0,...,0.337654,3.0,15.0,0.200000,11.400000,22.000000,0.517610,7.533333,15.000000,0.502222
6,2000.03,Chiyotaikai,Kokonoe,Oita,23.10,Maegashira,181.5,158.5,7.334247,3.0,...,0.600292,9.0,15.0,0.600000,13.200000,21.266667,0.612710,8.933333,14.266667,0.620000
7,2000.03,Chiyotenzan,Kokonoe,Osaka,24.10,Maegashira,183.5,150.0,9.008219,14.0,...,0.567334,9.0,15.0,0.600000,11.266667,21.266667,0.519796,7.200000,14.266667,0.504444
8,2000.03,Daishi,Oshiogawa,Ibaraki,31.60,Juryo,182.0,168.5,16.010959,36.0,...,0.429155,5.0,15.0,0.333333,10.200000,21.066667,0.451240,6.600000,14.066667,0.440000
9,2000.03,Daizen,Nishonoseki,Osaka,35.20,Maegashira,188.3,160.0,19.013699,21.0,...,0.623234,10.0,15.0,0.666667,10.466667,22.000000,0.471735,6.600000,15.000000,0.440000
